# 第10课 - 生产环境中的 AI 代理

在本课中，您将学习使用 Microsoft Agent Framework (Python) 的 AI 代理<strong>生产模式</strong>。我们涵盖：

- <strong>可观测性</strong> — 为代理交互添加计时和日志记录
- <strong>评估</strong> — 使用评估代理为响应质量打分
- <strong>成本管理</strong> — 代币优化和模型选择的策略

场景是一个帮助用户规划旅行的<strong>旅行代理</strong>，并在其上叠加监控和评估。


## 设置


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## 生产考虑

将 AI 代理从原型转移到生产环境需要仔细关注三个支柱：

1. <strong>可观察性</strong> — 你需要了解代理在做什么，耗时多久，以及调用了哪些工具。没有跟踪和日志，调试生产问题几乎不可能。

2. <strong>评估</strong> — 自动化质量检查确保代理的响应随着时间保持准确、完整且有帮助。评估代理可以根据定义的标准对响应进行评分。

3. <strong>成本管理</strong> — 令牌使用直接影响成本。诸如提示优化、模型选择和缓存等策略有助于在不牺牲质量的前提下控制开支。


## 构建一个可观察代理

我们定义了旅行工具并用计时包装了代理调用，以便我们能够监控延迟。在生产环境中，您会集成OpenTelemetry或类似的追踪后端。


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## 评估模式

一个常见的生产模式是使用第二个代理作为<strong>评估者</strong>。评估者根据预定义的标准（如完整性、准确性和有用性）对主代理的响应进行评分。

这可以实现：
- 在响应传达给用户之前自动进行质量把关
- 在提示词或模型发生变化时检测回归
- 持续监控代理的性能表现


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## 成本管理策略

控制成本对于生产环境中的 AI 代理至关重要。以下是关键策略：

| 策略 | 描述 |
|---|---|
| <strong>提示优化</strong> | 保持系统指令简洁。移除冗余上下文以减少输入令牌数。 |
    "| <strong>模型选择</strong> | 对于分类或抽取等简单任务使用更小、更便宜的模型（如 GPT-5-mini），将复杂推理任务留给更强大的模型。 |\n",
| <strong>缓存</strong> | 缓存工具结果和常见查询，避免重复调用 API。 |
| <strong>令牌预算</strong> | 设置 `max_tokens` 限制，防止回复意外过长。 |
| <strong>批处理</strong> | 尽可能将多个用户查询合并为一次 API 调用。 |

在实际操作中，分层方法效果良好：将简单请求路由到快速且廉价的模型，仅将复杂查询升级到更强大的模型。


## 总结

在本课中，您学习了如何：

1. <strong>为代理交互添加可观察性</strong>，包括时间记录和日志，为跟踪和监控打下基础。
2. <strong>自动评估代理响应</strong>，使用评估代理对完整性、准确性和有用性进行评分。
3. <strong>管理成本</strong>，通过提示优化、模型选择、缓存和令牌预算实现。

这些生产模式有助于确保您的AI代理在大规模应用中可靠、可衡量且具有成本效益。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文件由 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻译完成。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言版文件应视为权威来源。对于重要信息，建议使用专业人工翻译。我们对因使用本翻译而产生的任何误解或误释不承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
